# 01 - Dataset Setup

Downloads the fracture detection dataset from Roboflow in YOLOv8 format. The downloaded dataset is ready to use — no extra processing needed.

Works on local, Colab (with Drive), and Kaggle.

In [ ]:
RUN_ENV = "local"       # "local" | "colab" | "kaggle"
PROJECT_NAME = "yolov8-fracture-detection"

In [ ]:
from pathlib import Path

if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
elif RUN_ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working") / PROJECT_NAME
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DATA_ROOT = PROJECT_ROOT / "data"
RUNS_ROOT = PROJECT_ROOT / "runs"
WEIGHTS_ROOT = PROJECT_ROOT / "weights"

for folder in (DATA_ROOT, RUNS_ROOT, WEIGHTS_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_ROOT}")

## Download dataset from Roboflow

Installs the Roboflow package and downloads the dataset into `data/fracture/` inside the project root. On Colab this lands in your Drive folder so it persists.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("abdirisak-olol").project("bone-fracture-7fylg-inppu")
version = project.version(1)
dataset = version.download("yolov8", location=str(DATA_ROOT / "fracture"))

DATA_YAML = next(Path(dataset.location).glob("*.yaml"))
print(f"Dataset location : {dataset.location}")
print(f"Dataset YAML     : {DATA_YAML}")

In [3]:
# Examples:
# Colab Drive: Path("/content/drive/MyDrive/fracture-data/export")
# Kaggle:      Path("/kaggle/input/my-fracture-yolo-dataset")
# Local:       RAW_ROOT / "my-fracture-yolo-export"
SOURCE_DATASET = RAW_ROOT / "my-fracture-yolo-export"


## Option B - Optional Roboflow download

Use this only if you have a Roboflow project/version exported as YOLO. Store the API key outside the repo:

- Colab: put it in Colab secrets or set it in the session.
- Kaggle: use Kaggle notebook secrets.
- Local: use an untracked `.env` or shell environment variable.

The export should be a fracture-only object detection version. Do not commit Roboflow download URLs or API keys.


In [ ]:
USE_ROBOFLOW = False
ROBOFLOW_WORKSPACE = "your-workspace"
ROBOFLOW_PROJECT = "your-project"
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = "yolov8"  # Use the YOLOv8 export when available; YOLOv5 PyTorch is also YOLO-compatible.

if USE_ROBOFLOW:
    api_key = os.environ.get("ROBOFLOW_API_KEY")
    if not api_key:
        raise RuntimeError("Set ROBOFLOW_API_KEY in your notebook secrets or environment first.")
    from roboflow import Roboflow

    rf = Roboflow(api_key=api_key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    downloaded = version.download(model_format=ROBOFLOW_FORMAT, location=str(RAW_ROOT / "roboflow-fracture"))
    SOURCE_DATASET = Path(downloaded.location)

SOURCE_DATASET


## Build the small single-class subset

The source should already be fracture-focused. If your source has multiple medical labels, filter it first in the source system or create a fracture-only Roboflow version before running this cell.


In [4]:
import random
import shutil
from dataclasses import dataclass

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

@dataclass(frozen=True)
class DatasetCounts:
    train_images: int
    val_images: int
    test_images: int
    train_labels: int
    val_labels: int
    test_labels: int


def write_fracture_yaml(dataset_root, yaml_path=None):
    root = Path(dataset_root).expanduser().resolve()
    target = Path(yaml_path).expanduser().resolve() if yaml_path else root / "fracture.yaml"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        "\n".join([
            f"path: {root.as_posix()}",
            "train: images/train",
            "val: images/val",
            "test: images/test",
            "names:",
            "  0: fracture",
            "",
        ]),
        encoding="utf-8",
    )
    return target


def summarize_yolo_dataset(dataset_root):
    root = Path(dataset_root).expanduser()
    counts = []
    for kind in ("images", "labels"):
        for split in ("train", "val", "test"):
            folder = root / kind / split
            if kind == "images":
                counts.append(sum(1 for item in folder.glob("*") if item.suffix.lower() in IMAGE_EXTENSIONS))
            else:
                counts.append(sum(1 for item in folder.glob("*.txt")))
    return DatasetCounts(*counts)


def validate_single_class_labels(dataset_root):
    root = Path(dataset_root).expanduser()
    bad_lines = []
    for label_path in sorted((root / "labels").glob("**/*.txt")):
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
            stripped = line.strip()
            if not stripped:
                continue
            parts = stripped.split()
            if len(parts) != 5 or parts[0] != "0":
                bad_lines.append(f"{label_path}:{line_number}: {stripped}")
    if bad_lines:
        preview = "\n".join(bad_lines[:10])
        raise ValueError(f"Labels must be YOLO detect rows with class id 0 only:\n{preview}")


def candidate_split_dirs(source, split):
    source = Path(source)
    return (
        (source / "images" / split, source / "labels" / split),
        (source / split / "images", source / split / "labels"),
    )


def collect_labeled_images(source_root):
    source = Path(source_root).expanduser()
    examples = []
    for split in ("train", "valid", "val", "test"):
        for image_dir, label_dir in candidate_split_dirs(source, split):
            if not image_dir.exists() or not label_dir.exists():
                continue
            for image_path in image_dir.iterdir():
                if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                label_path = label_dir / f"{image_path.stem}.txt"
                if label_path.exists() and label_path.read_text(encoding="utf-8").strip():
                    examples.append((image_path, label_path))
    return examples


def split_examples(examples, val_fraction, test_fraction):
    total = len(examples)
    test_count = max(1, round(total * test_fraction))
    val_count = max(1, round(total * val_fraction))
    if test_count + val_count >= total:
        test_count = 1
        val_count = 1
    test_items = examples[:test_count]
    val_items = examples[test_count:test_count + val_count]
    train_items = examples[test_count + val_count:]
    return train_items, val_items, test_items


def copy_label_as_fracture_only(source, target, remap_all_labels_to_fracture=True):
    rows = []
    for line in Path(source).read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        if len(parts) != 5:
            raise ValueError(f"Expected YOLO detect label with 5 columns in {source}: {line}")
        if parts[0] != "0" and not remap_all_labels_to_fracture:
            raise ValueError(f"Found class id {parts[0]} in {source}; expected class id 0 only")
        rows.append(" ".join(["0", *parts[1:]]))
    Path(target).write_text("\n".join(rows) + ("\n" if rows else ""), encoding="utf-8")


def prepare_small_fracture_subset(
    source_root,
    output_root,
    max_images=240,
    seed=42,
    val_fraction=0.2,
    test_fraction=0.1,
    remap_all_labels_to_fracture=True,
):
    source = Path(source_root).expanduser()
    output = Path(output_root).expanduser()
    if not source.exists():
        raise FileNotFoundError(f"Source dataset does not exist: {source}")
    if not 0 <= val_fraction < 1 or not 0 <= test_fraction < 1 or val_fraction + test_fraction >= 1:
        raise ValueError("val_fraction and test_fraction must be non-negative and sum to less than 1")
    if max_images < 3:
        raise ValueError("max_images must be at least 3 so train/val/test can be populated")

    examples = collect_labeled_images(source)
    if len(examples) < 3:
        raise ValueError("Need at least three labeled fracture images to create train/val/test splits")

    random.Random(seed).shuffle(examples)
    selected = examples[:min(max_images, len(examples))]
    train_items, val_items, test_items = split_examples(selected, val_fraction, test_fraction)

    if output.exists():
        shutil.rmtree(output)
    for split in ("train", "val", "test"):
        (output / "images" / split).mkdir(parents=True, exist_ok=True)
        (output / "labels" / split).mkdir(parents=True, exist_ok=True)

    for split, items in (("train", train_items), ("val", val_items), ("test", test_items)):
        for image_path, label_path in items:
            shutil.copy2(image_path, output / "images" / split / image_path.name)
            copy_label_as_fracture_only(
                label_path,
                output / "labels" / split / f"{image_path.stem}.txt",
                remap_all_labels_to_fracture,
            )

    return write_fracture_yaml(output)


In [ ]:
dataset_root = Path(dataset.location)
print(DATA_YAML.read_text())
for split in ("train", "valid", "test"):
    img_dir = dataset_root / split / "images"
    count = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    print(f"  {split}: {count} images")

## Next step

Open `02_train_yolov8_fracture.ipynb`. The `DATA_YAML` path is already configured to match — just set `RUN_ENV` to the same value and run.

## Next step

Open `02_train_yolov8_fracture.ipynb`, use the same `RUN_ENV`, and point `DATA_YAML` to the printed YAML path.
